In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import os
import warnings
warnings.filterwarnings('ignore')

# Load dataset
base = os.path.expanduser("~") + r"\phishguard-ai\datasets\phishing_urls.csv"
df = pd.read_csv(base)

print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['phishing'].value_counts()}")

Dataset shape: (88647, 112)
Label distribution:
phishing
0    58000
1    30647
Name: count, dtype: int64


In [5]:
# X = features (everything except the label)
# y = label (what we're predicting)
X = df.drop('phishing', axis=1)
y = df['phishing']

print(f"Features shape: {X.shape}")
print(f"Label shape: {y.shape}")
print(f"\nFeature names (first 10): {X.columns[:10].tolist()}")

Features shape: (88647, 111)
Label shape: (88647,)

Feature names (first 10): ['qty_dot_url', 'qty_hyphen_url', 'qty_underline_url', 'qty_slash_url', 'qty_questionmark_url', 'qty_equal_url', 'qty_at_url', 'qty_and_url', 'qty_exclamation_url', 'qty_space_url']


In [6]:
# Split: 80% train, 20% test
# random_state=42 means the split is reproducible — same split every run
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y    # ensures same phishing ratio in both splits
)

print(f"Training set: {X_train.shape[0]} URLs")
print(f"Test set:     {X_test.shape[0]} URLs")
print(f"\nTraining label distribution:")
print(y_train.value_counts())
print(f"\nTest label distribution:")
print(y_test.value_counts())

Training set: 70917 URLs
Test set:     17730 URLs

Training label distribution:
phishing
0    46400
1    24517
Name: count, dtype: int64

Test label distribution:
phishing
0    11600
1     6130
Name: count, dtype: int64


In [7]:
# n_estimators = number of trees
# n_jobs=-1 = use all CPU cores
# random_state=42 = reproducible results
print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)
print("Training complete.")

Training Random Forest...
Training complete.


In [8]:
# Make predictions on test set (data the model NEVER saw)
y_pred = rf_model.predict(X_test)

# Core metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("=" * 40)
print("RANDOM FOREST RESULTS")
print("=" * 40)
print(f"Accuracy:  {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Legitimate', 'Phishing']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

RANDOM FOREST RESULTS
Accuracy:  0.9706  (97.06%)
Precision: 0.9558
Recall:    0.9594
F1 Score:  0.9576

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.98      0.98      0.98     11600
    Phishing       0.96      0.96      0.96      6130

    accuracy                           0.97     17730
   macro avg       0.97      0.97      0.97     17730
weighted avg       0.97      0.97      0.97     17730


Confusion Matrix:
[[11328   272]
 [  249  5881]]


In [9]:
# Feature importance — which of the 111 features does the model rely on most?
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 most important features:")
print(importance_df.head(20).to_string(index=False))

Top 20 most important features:
                feature  importance
       directory_length    0.118937
 time_domain_activation    0.061837
   qty_dollar_directory    0.044969
           qty_dot_file    0.043723
             length_url    0.038558
    qty_slash_directory    0.033838
          qty_slash_url    0.028222
           ttl_hostname    0.025703
qty_underline_directory    0.024119
                 asn_ip    0.023757
            file_length    0.023446
  qty_percent_directory    0.023384
 time_domain_expiration    0.022954
          time_response    0.022797
    qty_comma_directory    0.022410
            qty_at_file    0.022256
      qty_and_directory    0.022195
          domain_length    0.019237
         qty_dot_domain    0.017303
       qty_at_directory    0.017075
